In [ ]:
import pandas as pd
import numpy as np
import os

print("开始计算产品层面统计...")
os.chdir(r'G:\Kuangyu_Temp\Outsource')
# 1. 读取数据
df = pd.read_stata('lenth9.dta')
print(f"原始数据: {len(df):,} 行")

# 2. 提前筛选：只保留既买又卖的产品（外包产品）
buy_set = set(df[df['is_output']==0][['firm_id','year','product_id']].itertuples(index=False, name=None))
sell_set = set(df[df['is_output']==1][['firm_id','year','product_id']].itertuples(index=False, name=None))
outsource_set = buy_set & sell_set
df['key'] = list(zip(df['firm_id'], df['year'], df['product_id']))
df = df[df['key'].isin(outsource_set)].copy()
df.drop('key', axis=1, inplace=True)
print(f"筛选后: {len(df):,} 行（只保留外包产品）")

# 3. 分离购入和销售
buy = df[df['is_output']==0][['firm_id','year','product_id','v']].copy()
buy.columns = ['firm_id','year','product_id','input']
sell = df[df['is_output']==1][['firm_id','year','product_id','v']].copy()
sell.columns = ['firm_id','year','product_id','output']

# 4. 合并并计算外包/自产
prod = sell.merge(buy, on=['firm_id','year','product_id'], how='left').fillna(0)
prod['outsourcing_output'] = prod[['input','output']].min(axis=1)
prod['production_output'] = (prod['output'] - prod['outsourcing_output']).clip(lower=0)
prod['total_output'] = prod['output']

# 5. 企业层面汇总（向量化）
firm_agg = prod.groupby(['firm_id','year']).agg({
    'total_output': 'sum',
    'outsourcing_output': 'sum',
    'production_output': 'sum'
}).reset_index()
firm_agg.columns = ['firm_id','year','firm_total','firm_outsourcing','firm_production']

prod = prod.merge(firm_agg, on=['firm_id','year'])

# 6. 计算排名和主营产品（向量化）
prod['output_ranking'] = prod.groupby(['firm_id','year'])['total_output'].rank(method='first', ascending=False).astype(int)
main = prod[prod['output_ranking']==1][['firm_id','year','product_id','total_output']]
main.columns = ['firm_id','year','main_product','main_output']

prod = prod.merge(main, on=['firm_id','year'])

# 7. 计算所有比例（向量化）
prod['sales_percen'] = prod['total_output'] / prod['main_output']
prod['production_percen'] = np.where(prod['total_output']>0,
                                      prod['production_output']/prod['total_output'], 0)
prod['output_share'] = prod['total_output'] / prod['firm_total']
prod['outsourcing_share'] = np.where(prod['firm_outsourcing']>0,
                                      prod['outsourcing_output']/prod['firm_outsourcing'], 0)
prod['production_share'] = np.where(prod['firm_production']>0,
                                     prod['production_output']/prod['firm_production'], 0)

# 8. 输出最终结果
result = prod[[
    'firm_id', 'year', 'product_id', 'output_ranking', 'main_product',
    'total_output', 'outsourcing_output', 'production_output',
    'sales_percen', 'production_percen',
    'output_share', 'outsourcing_share', 'production_share'
]].copy()
result.rename(columns={'year':'year'}, inplace=True)

result.to_stata(r'G:\Kuangyu_Temp\Outsource\description\product_level_analysis.dta', write_index=False)
print(f"完成！共 {len(result):,} 条产品记录")
print(result.head(20))

开始计算产品层面统计...
原始数据: 465,487,031 行


KeyError: "['date'] not in index"

In [2]:
df.head()

,firm_id,year,input_output,product_id,v
0,-122,2018.0,input,107060199000000,57124.000000
1,-122,2018.0,input,109013004000000,264000.000000
2,-122,2017.0,output,109069900000000,473850.000000
3,-122,2017.0,input,110010102020000,2836.500000
4,-122,2018.0,input,110010102020000,2760.599854
